# Trim or remove one speaker from call recordings on Kaggle

This notebook is designed for call recordings uploaded as a Kaggle Dataset. It does **not** use Whisper, WhisperX, or NVIDIA NeMo.

It tries the cheap path first: inspect whether the call has separate stereo channels. If the call is mono or mixed, it uses `pyannote.audio` speaker diarization to find speaker time ranges, then exports either:

- only the selected speaker, or
- the call with that speaker removed.

For pyannote, you need a Hugging Face token and access to the gated pyannote models.

In [1]:
!pip -q install pyannote.audio pydub pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 644.0 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.5/893.5 kB 7.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 58.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

## Setup

1. Upload your folder of `.mpeg` files as a Kaggle Dataset.
2. Add a Kaggle secret named `HF_TOKEN` with your Hugging Face token.
3. On Hugging Face, accept access for:
   - `pyannote/speaker-diarization-3.1`
   - `pyannote/segmentation-3.0`

Set `INPUT_DIR` to the dataset folder Kaggle creates under `/kaggle/input`.

In [2]:
from pathlib import Path
import json
import os
import subprocess
from typing import Iterable

import pandas as pd
from pydub import AudioSegment
from tqdm.auto import tqdm

# Change this after attaching your Kaggle Dataset.
INPUT_DIR = Path('/kaggle/input/w-audio')
WORK_DIR = Path('/kaggle/working/speaker_trim')
SEGMENT_DIR = WORK_DIR / 'segments'
PREVIEW_DIR = WORK_DIR / 'speaker_previews'
OUTPUT_DIR = WORK_DIR / 'output_audio'

for directory in [WORK_DIR, SEGMENT_DIR, PREVIEW_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

AUDIO_EXTENSIONS = {'.wav', '.mp3', '.mpeg', '.mpg', '.m4a', '.ogg', '.opus', '.flac'}
audio_files = sorted(p for p in INPUT_DIR.rglob('*') if p.suffix.lower() in AUDIO_EXTENSIONS)

print(f'Found {len(audio_files)} audio files')
for path in audio_files[:10]:
    print(path)

Found 0 audio files


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## Inspect channels

If the calls are stereo and each speaker is isolated to a different channel, use channel extraction instead of diarization. That is much faster and cleaner. If they are mono, continue to pyannote diarization.

In [ ]:
def ffprobe_audio(path: Path) -> dict:
    cmd = [
        'ffprobe', '-v', 'error',
        '-select_streams', 'a:0',
        '-show_entries', 'stream=codec_name,channels,channel_layout,sample_rate,duration',
        '-of', 'json',
        str(path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    data = json.loads(result.stdout)
    return data.get('streams', [{}])[0]

probe_rows = []
for path in audio_files:
    info = ffprobe_audio(path)
    probe_rows.append({
        'file': path.name,
        'channels': info.get('channels'),
        'layout': info.get('channel_layout'),
        'sample_rate': info.get('sample_rate'),
        'duration': info.get('duration'),
        'codec': info.get('codec_name'),
    })

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(WORK_DIR / 'audio_probe.csv', index=False)
probe_df

In [ ]:
def extract_channel(input_path: Path, channel: str, output_path: Path):
    """channel must be 'left' or 'right'."""
    pan = 'c0=c0' if channel == 'left' else 'c0=c1'
    cmd = [
        'ffmpeg', '-y', '-i', str(input_path),
        '-af', f'pan=mono|{pan}',
        '-ar', '22050',
        str(output_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Use this only if probe_df shows channels == 2 and listening confirms Aswini is always on one side.
# CHANNEL_TO_KEEP = 'left'
# for path in tqdm(audio_files):
#     extract_channel(path, CHANNEL_TO_KEEP, OUTPUT_DIR / f'{path.stem}__{CHANNEL_TO_KEEP}.wav')

## Run pyannote diarization

This labels speakers as `SPEAKER_00`, `SPEAKER_01`, etc. Those IDs are local to each file. You must preview the speakers before choosing which one to keep/remove.

In [ ]:
import torch
from pyannote.audio import Pipeline

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError('Missing HF_TOKEN. Add it as a Kaggle secret or environment variable.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    # pyannote.audio >= 3 uses token=
    pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', token=HF_TOKEN)
except TypeError:
    # Older pyannote.audio versions used use_auth_token=
    pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', use_auth_token=HF_TOKEN)
pipeline.to(device)
print(f'Using device: {device}')

In [ ]:
def get_annotation(diarization_result):
    """Return a pyannote Annotation across pyannote API versions."""
    if hasattr(diarization_result, 'itertracks'):
        return diarization_result

    for attr in ['speaker_diarization', 'diarization', 'annotation']:
        value = getattr(diarization_result, attr, None)
        if hasattr(value, 'itertracks'):
            return value

    if isinstance(diarization_result, dict):
        for key in ['speaker_diarization', 'diarization', 'annotation']:
            value = diarization_result.get(key)
            if hasattr(value, 'itertracks'):
                return value

    raise TypeError(f'Unsupported diarization output type: {type(diarization_result)}')

def diarize_file(path: Path) -> pd.DataFrame:
    diarization = get_annotation(pipeline(str(path)))
    rows = []
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        rows.append({
            'file': path.name,
            'speaker': speaker,
            'start': float(turn.start),
            'end': float(turn.end),
            'duration': float(turn.end - turn.start),
        })
    return pd.DataFrame(rows)

def save_speaker_previews(path: Path, segments: pd.DataFrame, seconds_per_speaker: int = 20):
    audio = AudioSegment.from_file(path)
    for speaker, speaker_segments in segments.groupby('speaker'):
        preview = AudioSegment.empty()
        used_ms = 0
        for row in speaker_segments.sort_values('duration', ascending=False).itertuples():
            start_ms = max(0, int(row.start * 1000))
            end_ms = min(len(audio), int(row.end * 1000))
            clip = audio[start_ms:end_ms]
            preview += clip
            used_ms += len(clip)
            if used_ms >= seconds_per_speaker * 1000:
                break
        out = PREVIEW_DIR / f'{path.stem}__{speaker}.wav'
        preview.export(out, format='wav')

# Start with one file so you can inspect speaker labels before processing everything.
sample_file = audio_files[0]
sample_segments = diarize_file(sample_file)
sample_segments.to_csv(SEGMENT_DIR / f'{sample_file.stem}.csv', index=False)
save_speaker_previews(sample_file, sample_segments)
sample_segments.groupby('speaker')['duration'].sum().sort_values(ascending=False)

Listen to the files in `/kaggle/working/speaker_trim/speaker_previews`. Decide which label is the speaker you want for that recording.

Important: diarization labels can flip between files. `SPEAKER_00` in one recording is not guaranteed to be Aswini in another recording.

In [ ]:
# For a single-file test, set this after listening to the preview clips.
TARGET_SPEAKER_BY_FILE = {
    sample_file.name: 'SPEAKER_00',
}

# Export mode: 'keep' means only target speaker. 'remove' means delete target speaker parts.
MODE = 'keep'

# If MODE == 'remove': True keeps original duration by replacing target segments with silence.
PRESERVE_TIMING_WITH_SILENCE = False

# Padding helps avoid clipped syllables at segment edges.
PADDING_SEC = 0.15
MIN_SEGMENT_SEC = 0.25

In [ ]:
def normalized_intervals(segments: pd.DataFrame, target_speaker: str, audio_ms: int) -> list[tuple[int, int]]:
    intervals = []
    for row in segments[segments['speaker'] == target_speaker].itertuples():
        if row.duration < MIN_SEGMENT_SEC:
            continue
        start_ms = max(0, int((row.start - PADDING_SEC) * 1000))
        end_ms = min(audio_ms, int((row.end + PADDING_SEC) * 1000))
        if end_ms > start_ms:
            intervals.append((start_ms, end_ms))

    if not intervals:
        return []

    intervals.sort()
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end:
            merged[-1] = (prev_start, max(prev_end, end))
        else:
            merged.append((start, end))
    return merged

def export_trimmed(path: Path, segments: pd.DataFrame, target_speaker: str, mode: str):
    audio = AudioSegment.from_file(path)
    intervals = normalized_intervals(segments, target_speaker, len(audio))

    if mode == 'keep':
        output = AudioSegment.empty()
        for start_ms, end_ms in intervals:
            output += audio[start_ms:end_ms]
    elif mode == 'remove':
        output = AudioSegment.empty()
        cursor = 0
        for start_ms, end_ms in intervals:
            output += audio[cursor:start_ms]
            if PRESERVE_TIMING_WITH_SILENCE:
                output += AudioSegment.silent(duration=end_ms - start_ms, frame_rate=audio.frame_rate)
            cursor = end_ms
        output += audio[cursor:]
    else:
        raise ValueError("mode must be 'keep' or 'remove'")

    out = OUTPUT_DIR / f'{path.stem}__{mode}_{target_speaker}.wav'
    output.export(out, format='wav')
    return out

export_trimmed(sample_file, sample_segments, TARGET_SPEAKER_BY_FILE[sample_file.name], MODE)

## Batch processing

After confirming the first file, run diarization for all files. This writes per-file CSVs and preview clips. Then fill `TARGET_SPEAKER_BY_FILE` for each recording and export.

In [ ]:
all_segments = []

for path in tqdm(audio_files):
    csv_path = SEGMENT_DIR / f'{path.stem}.csv'
    if csv_path.exists():
        segments = pd.read_csv(csv_path)
    else:
        segments = diarize_file(path)
        segments.to_csv(csv_path, index=False)
        save_speaker_previews(path, segments)
    all_segments.append(segments)

all_segments_df = pd.concat(all_segments, ignore_index=True)
all_segments_df.to_csv(WORK_DIR / 'all_segments.csv', index=False)

summary = all_segments_df.groupby(['file', 'speaker'])['duration'].sum().reset_index()
summary.to_csv(WORK_DIR / 'speaker_duration_summary.csv', index=False)
summary

In [ ]:
# Fill this after listening to preview clips for each file.
# Example:
# TARGET_SPEAKER_BY_FILE = {
#     'amarnath - aswini.mpeg': 'SPEAKER_01',
#     'anup menon - aswini.mpeg': 'SPEAKER_00',
# }

missing = [path.name for path in audio_files if path.name not in TARGET_SPEAKER_BY_FILE]
if missing:
    print('Missing target speaker labels for:')
    for name in missing:
        print(' -', name)
else:
    exported = []
    for path in tqdm(audio_files):
        segments = pd.read_csv(SEGMENT_DIR / f'{path.stem}.csv')
        exported.append(export_trimmed(path, segments, TARGET_SPEAKER_BY_FILE[path.name], MODE))
    print('Exported files:')
    for path in exported:
        print(path)

## Download results

The final audio files are under `/kaggle/working/speaker_trim/output_audio`. You can zip them from the Kaggle file browser or run the cell below.

In [ ]:
!cd /kaggle/working && zip -qr speaker_trim_results.zip speaker_trim
print('/kaggle/working/speaker_trim_results.zip')